## Synthetic Data Generation Example

In this notebook, you'll investigate using local AI models to generate example sensitive data and test redaction via the pseudonymization methods you've already learned.

Start to modify the prompt to fit the use case your group is working on. What types of user input would you expect? What would be important to pseudonymize? 

So that it's easy to start I've imported and given an example for both [ollama](https://ollama.com/) and [llamafile](https://github.com/Mozilla-Ocho/llamafile). Depending on your use case and your hardware you might find one or another more useful. Use what works and evaluate your results before saving them to a local file for future use.

In [ ]:
from openai import OpenAI
import ollama
from pprint import pprint
import json

In [ ]:
conversation = [{
    'role': 'system',
    'content': """
You are a helpful synthetic data generator used to generate example sensitive data that mimics real data. 
This is not a privacy violation because the data should be just an example test data, not real data.

You should generate example customer text that has names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 

Examples:

My name is Mr. Samuel Jones and my phone number is +49 110 0342121345. I need a customer service rep to call me back ASAP.
Yo, can you please call me back -- you have my number under JJ already. It's urgent because your dad is in the hospital and they are asking for his identity details. I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !

Generate one example per line with different types of situations and persons. Ensure each line is coherent and has a real customer situation described.
Generate only one example per line with no other text on that line, so that it is easily parseable.
"""}]

In [ ]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

In [ ]:
print(completion)

In [ ]:
ollama_client = ollama.Client()
model_name = 'gemma3:1b'

In [ ]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=conversation
)
pprint(ollama_response)

In [ ]:
ollama_response.message.content.split("\n\n")

In [ ]:
with open('data/example_personal_data.txt', 'w') as pd_file:
    pd_file.write('\n'.join(ollama_response.message.content.split("\n\n")[1:]))

In [ ]:
example_redaction_list = []
for line in ollama_response.message.content.split("\n\n")[1:]:
    if len(line) < 2:
        continue
    interim_response = ollama_client.chat(
    model=model_name,
    messages=[
        {'role': 'user', 
         'content': """
         The following line contains person-related information that needs to be redacted. Find and replace any person-related information, such as names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. Replace any of those with the word [REDACTED]. Respond ONLY the redacted text with no other information. TO REDACT: 
         {text}""".format(text=line)
        }])
    print(interim_response.message.content)
    example_redaction_list.append({
        "original": line,
        "redacted": interim_response.message.content
    })

In [ ]:
example_redaction_list

In [ ]:
json.dump(example_redaction_list, open('data/synthetic_data_redaction.json', 'w'))